In [1]:
import pandas as pd
import numpy as np
import sklearn

In [2]:
df_loan = pd.read_csv('/Users/sirawitjudjan/Documents/สมัครงาน/โปรเจคสมัครงาน/Credit risk analysis/loans_full_schema_1.csv')
df_loan = df_loan.dropna()  # Drop rows with missing values

In [3]:
x_col_names = ['log_annual_income', 'debt_to_income', 'delinq_2y', 'num_collections_last_12m', 'num_historical_failed_to_pay', 'num_cc_carrying_balance', 'account_never_delinq_percent', 'tax_liens', 'loan_amount', 'term', 'interest_rate', 'grade', 'total_credit_utilized']
y_col_name = 'loan_status'

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = df_loan[x_col_names].copy()
X = add_constant(X)
vif_df = pd.DataFrame()
vif_df["Feature"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]
print(vif_df[vif_df["Feature"] != "const"].reset_index(drop=True))

                         Feature        VIF
0              log_annual_income   1.939192
1                 debt_to_income   2.001324
2                      delinq_2y   1.218766
3       num_collections_last_12m   1.024169
4   num_historical_failed_to_pay   1.476892
5        num_cc_carrying_balance   1.167912
6   account_never_delinq_percent   1.241408
7                      tax_liens   1.497180
8                    loan_amount   1.573822
9                           term   1.446028
10                 interest_rate  19.583046
11                         grade  19.571409
12         total_credit_utilized   2.167709


In [5]:
x_col_names = ['log_annual_income', 'debt_to_income', 'delinq_2y', 'num_collections_last_12m', 'num_historical_failed_to_pay', 'num_cc_carrying_balance', 'account_never_delinq_percent', 'tax_liens', 'loan_amount', 'term', 'interest_rate', 'total_credit_utilized']


In [6]:
from sklearn.model_selection import train_test_split
X = df_loan[x_col_names]
y = df_loan[y_col_name]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

In [9]:
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)

In [10]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8073394495412844


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[88  0]
 [21  0]]
              precision    recall  f1-score   support

           0       0.81      1.00      0.89        88
           1       0.00      0.00      0.00        21

    accuracy                           0.81       109
   macro avg       0.40      0.50      0.45       109
weighted avg       0.65      0.81      0.72       109



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [12]:
from sklearn.metrics import roc_auc_score

y_prob = log_reg.predict_proba(X_test_scaled)[:,1]

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC =", auc)


ROC-AUC = 0.6471861471861471
